# Premier League Match Analysis: VAR and Machine Learning

This notebook explores pre-/post-VAR match statistics, evaluates models trained on pre-VAR matches against post-VAR matches, and interprets model predictions. The analysis is observational: it does not establish that VAR caused any differences.

## 1. Setup and data loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully")

In [ ]:
# Load the dataset
from pathlib import Path
import pandas as pd

filename = "Dataset1_Premier_League_1993_2024.csv"
possible_paths = [
    Path(filename),                    # Same folder as this notebook
    Path("data") / filename,           # Optional data/ folder
    Path("/content/sample_data") / filename,  # Original Google Colab location
]

data_path = next((path for path in possible_paths if path.is_file()), None)

# In Google Colab, offer an upload if the CSV was not found.
if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        if filename in uploaded:
            data_path = Path(filename)
    except ImportError:
        pass

if data_path is None or not data_path.is_file():
    raise FileNotFoundError(
        f"Could not find '{filename}'. Save it beside this notebook or in a "
        "'data/' folder. In Google Colab, run this cell and upload the CSV "
        "when prompted."
    )

df = pd.read_csv(data_path)
print(f"Dataset loaded: {data_path}")
print(f"Raw dataset shape: {df.shape}")
display(df.head())

## 2. Data quality checks

Inspect the dataset structure, variable types and missingness before defining the analysis sample.

In [ ]:
# Check column names and data types
print("Columns and data types:")
print(df.dtypes)

print("\nBasic statistics:")
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Values': missing,
    'Percentage': missing_percent
})

print(missing_summary[missing_summary['Missing Values'] > 0])

In [ ]:
# Check which seasons have missing data
missing_by_season = df[df['H Shots'].isnull()]['Season'].unique()
print("Seasons with missing shot/card/foul data:")
print(sorted(missing_by_season))

## 3. Cleaning and defining the VAR periods

Prepare the analysis sample and label seasons before and after the 2019/20 introduction of VAR.

In [ ]:
# Remove seasons with missing data (1993/94 to 1999/2000)
df = df[~df['Season'].isin(['1993/94', '1994/95', '1995/96',
                             '1996/97', '1997/98', '1998/99', '1999/00'])]

# Reset the index
df = df.reset_index(drop=True)

# Confirm the result
print("Seasons remaining:", sorted(df['Season'].unique()))
print("Total matches remaining:", len(df))
print("\nMissing values after filtering:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Define post-VAR seasons
post_var_seasons = ['2019/20', '2020/21', '2021/22',
                    '2022/23', '2023/24', '2024/25']

# Add VAR flag (0 = pre-VAR, 1 = post-VAR)
df['VAR'] = df['Season'].apply(lambda x: 1 if x in post_var_seasons else 0)

# Add COVID flag (2020/21 season played without crowds)
df['COVID'] = df['Season'].apply(lambda x: 1 if x == '2020/21' else 0)

# Confirm
print("VAR flag distribution:")
print(df['VAR'].value_counts())
print()
print("COVID flag distribution:")
print(df['COVID'].value_counts())
print()
print("Pre-VAR matches:", len(df[df['VAR'] == 0]))
print("Post-VAR matches:", len(df[df['VAR'] == 1]))

## 4. Exploratory analysis

Review outcome distributions and compare selected match statistics across the two periods.

In [ ]:
# Check what values are in the FT Result column
print("FT Result values:")
print(df['FT Result'].value_counts())

In [ ]:
# Compare FT Result distribution pre vs post VAR
print("Pre-VAR results:")
print(df[df['VAR']==0]['FT Result'].value_counts(normalize=True).round(3) * 100)
print()
print("Post-VAR results:")
print(df[df['VAR']==1]['FT Result'].value_counts(normalize=True).round(3) * 100)

In [ ]:
# Visualise the result distribution pre vs post VAR
import matplotlib.pyplot as plt
import numpy as np

categories = ['Home Win', 'Away Win', 'Draw']
pre_var = [46.5, 28.3, 25.2]
post_var = [43.7, 33.0, 23.3]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, pre_var, width, label='Pre-VAR', color='steelblue')
bars2 = ax.bar(x + width/2, post_var, width, label='Post-VAR', color='coral')

ax.set_xlabel('Match Outcome')
ax.set_ylabel('Percentage (%)')
ax.set_title('Match Outcome Distribution: Pre-VAR vs Post-VAR')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylim(0, 60)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height()}%', ha='center', va='bottom')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height()}%', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('match_outcome_distribution.png', dpi=300)
plt.show()
print("Chart saved!")

In [ ]:
# Compare key match variables pre vs post VAR
variables = ['FTH Goals', 'FTA Goals', 'H Yellow', 'A Yellow',
             'H Red', 'A Red', 'H Fouls', 'A Fouls']

pre_var = df[df['VAR']==0][variables].mean()
post_var = df[df['VAR']==1][variables].mean()

comparison = pd.DataFrame({
    'Pre-VAR Average': pre_var.round(3),
    'Post-VAR Average': post_var.round(3),
    'Change': (post_var - pre_var).round(3),
    'Change %': ((post_var - pre_var) / pre_var * 100).round(1)
})

print(comparison)

In [ ]:
# Visualise key variable changes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Key Match Variables: Pre-VAR vs Post-VAR',
             fontsize=16, fontweight='bold')

# Goals
ax1 = axes[0, 0]
goals_pre = [1.530, 1.137]
goals_post = [1.559, 1.325]
x = np.arange(2)
ax1.bar(x - 0.175, goals_pre, 0.35, label='Pre-VAR', color='steelblue')
ax1.bar(x + 0.175, goals_post, 0.35, label='Post-VAR', color='coral')
ax1.set_title('Average Goals Per Match')
ax1.set_xticks(x)
ax1.set_xticklabels(['Home Goals', 'Away Goals'])
ax1.legend()
ax1.set_ylabel('Average per match')

# Cards
ax2 = axes[0, 1]
cards_pre = [1.394, 1.760, 0.064, 0.093]
cards_post = [1.697, 1.873, 0.055, 0.062]
x = np.arange(4)
ax2.bar(x - 0.175, cards_pre, 0.35, label='Pre-VAR', color='steelblue')
ax2.bar(x + 0.175, cards_post, 0.35, label='Post-VAR', color='coral')
ax2.set_title('Average Cards Per Match')
ax2.set_xticks(x)
ax2.set_xticklabels(['H Yellow', 'A Yellow', 'H Red', 'A Red'])
ax2.legend()
ax2.set_ylabel('Average per match')

# Fouls
ax3 = axes[1, 0]
fouls_pre = [11.486, 12.058]
fouls_post = [10.678, 10.835]
x = np.arange(2)
ax3.bar(x - 0.175, fouls_pre, 0.35, label='Pre-VAR', color='steelblue')
ax3.bar(x + 0.175, fouls_post, 0.35, label='Post-VAR', color='coral')
ax3.set_title('Average Fouls Per Match')
ax3.set_xticks(x)
ax3.set_xticklabels(['Home Fouls', 'Away Fouls'])
ax3.legend()
ax3.set_ylabel('Average per match')
ax3.set_ylim(9, 14)

# Change percentage
ax4 = axes[1, 1]
changes = [1.9, 16.6, 21.7, 6.4, -13.9, -33.8, -7.0, -10.1]
labels = ['H Goals', 'A Goals', 'H Yellow', 'A Yellow',
          'H Red', 'A Red', 'H Fouls', 'A Fouls']
colors = ['green' if x > 0 else 'red' for x in changes]
ax4.barh(labels, changes, color=colors)
ax4.axvline(x=0, color='black', linewidth=0.8)
ax4.set_title('Percentage Change After VAR (%)')
ax4.set_xlabel('Change (%)')

plt.tight_layout()
plt.savefig('key_variables_comparison.png', dpi=300)
plt.show()
print("Chart saved!")

## 5. Statistical analysis

Use complementary tests and regression models to examine differences in selected match statistics. Interpret these as associations rather than causal effects.

In [ ]:
# ============================================================
# CELL A — Combined significance testing (Welch's t-test + Mann-Whitney U)
# Replaces/extends the existing Cell 12
# ============================================================
from scipy import stats

print("Statistical Testing: Pre-VAR vs Post-VAR")
print("=" * 70)

variables = {
    'Away Goals': 'FTA Goals',
    'Home Goals': 'FTH Goals',
    'Home Yellow Cards': 'H Yellow',
    'Away Yellow Cards': 'A Yellow',
    'Home Red Cards': 'H Red',
    'Away Red Cards': 'A Red',
    'Home Fouls': 'H Fouls',
    'Away Fouls': 'A Fouls'
}

pre = df[df['VAR']==0]
post = df[df['VAR']==1]

results = []
for name, col in variables.items():
    u_stat, p_mw = stats.mannwhitneyu(pre[col], post[col], alternative='two-sided')
    t_stat, p_t = stats.ttest_ind(pre[col], post[col], equal_var=False)  # Welch's t-test
    results.append({
        'Variable': name,
        'Pre-VAR Mean': round(pre[col].mean(), 3),
        'Post-VAR Mean': round(post[col].mean(), 3),
        'Welch t P-value': round(p_t, 4),
        'Mann-Whitney U P-value': round(p_mw, 4),
        'Significant (both tests)': 'Yes' if (p_t < 0.05 and p_mw < 0.05) else 'No'
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


# ============================================================
# CELL B — Negative Binomial GLM (VAR as predictor) for count variables
# New cell, insert after Cell A
# ============================================================
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

print("Negative Binomial GLM: VAR as predictor")
print("=" * 70)

glm_results = []
for name, col in variables.items():
    formula = f'Q("{col}") ~ VAR'
    model = smf.negativebinomial(formula=formula, data=df)
    fit = model.fit(disp=0)
    coef = fit.params['VAR']
    pval = fit.pvalues['VAR']
    irr = np.exp(coef)  # incidence rate ratio: exp(coefficient)
    glm_results.append({
        'Variable': name,
        'VAR Coefficient': round(coef, 4),
        'Incidence Rate Ratio': round(irr, 3),
        'P-Value': round(pval, 4),
        'Significant': 'Yes' if pval < 0.05 else 'No'
    })

glm_df = pd.DataFrame(glm_results)
print(glm_df.to_string(index=False))
print("\nInterpretation: IRR > 1 means the rate increases post-VAR (e.g. IRR of 1.166")
print("for Away Goals means a 16.6% increase); IRR < 1 means a decrease.")


# ============================================================
# CELL C — Multinomial logistic regression for match outcome (H/A/D)
# New cell, insert after Cell B
# ============================================================
import statsmodels.api as sm

print("Multinomial Logistic Regression: Match Outcome ~ VAR")
print("=" * 70)

# Draw (D) set as the reference/baseline category
result_map = {'D': 0, 'H': 1, 'A': 2}
df['Result_Code'] = df['FT Result'].map(result_map)

X = sm.add_constant(df['VAR'])
y = df['Result_Code']

mnlogit_model = sm.MNLogit(y, X)
mnlogit_fit = mnlogit_model.fit(disp=0)
print(mnlogit_fit.summary())

print("\nOdds Ratios (relative to Draw baseline):")
odds_ratios = np.exp(mnlogit_fit.params)
odds_ratios.columns = ['Home Win vs Draw', 'Away Win vs Draw']
print(odds_ratios)

print("\nP-values:")
pvals = mnlogit_fit.pvalues
pvals.columns = ['Home Win vs Draw', 'Away Win vs Draw']
print(pvals)


In [ ]:
# ============================================================
# ROBUSTNESS CHECK 3 (RQ1): Recent-seasons-only statistical testing
# Insert after Cell C (multinomial logistic regression) and before Cell 14
# Mirrors the "Recent-seasons-only split" logic already used for RQ2/RQ3
# (Cell 23), but applied to RQ1's statistical tests instead of the ML models.
# ============================================================
import warnings
warnings.filterwarnings('ignore')

# Same balanced window used in the ML robustness check
recent_pre_var_seasons = ['2014/15', '2015/16', '2016/17', '2017/18', '2018/19']
recent_post_var_seasons = ['2019/20', '2020/21', '2021/22', '2022/23', '2023/24']

df_recent = df[df['Season'].isin(recent_pre_var_seasons + recent_post_var_seasons)].copy()

print("ROBUSTNESS CHECK 3 (RQ1): RECENT-SEASONS-ONLY STATISTICAL TESTING")
print("=" * 70)
print("Recent-window matches:", len(df_recent))
print(df_recent['VAR'].value_counts())

pre_r = df_recent[df_recent['VAR'] == 0]
post_r = df_recent[df_recent['VAR'] == 1]

# --- Welch's t-test + Mann-Whitney U (recent window) ---
print("\nWelch's t-test + Mann-Whitney U-test (recent-seasons window)")
print("-" * 70)
results_recent = []
for name, col in variables.items():
    u_stat, p_mw = stats.mannwhitneyu(pre_r[col], post_r[col], alternative='two-sided')
    t_stat, p_t = stats.ttest_ind(pre_r[col], post_r[col], equal_var=False)
    results_recent.append({
        'Variable': name,
        'Pre-VAR Mean': round(pre_r[col].mean(), 3),
        'Post-VAR Mean': round(post_r[col].mean(), 3),
        'Welch t P-value': round(p_t, 4),
        'Mann-Whitney U P-value': round(p_mw, 4),
        'Significant (both tests)': 'Yes' if (p_t < 0.05 and p_mw < 0.05) else 'No'
    })
results_recent_df = pd.DataFrame(results_recent)
print(results_recent_df.to_string(index=False))

# --- Negative Binomial GLM (recent window) ---
print("\nNegative Binomial GLM: VAR as predictor (recent-seasons window)")
print("-" * 70)
glm_recent_results = []
for name, col in variables.items():
    formula = f'Q("{col}") ~ VAR'
    fit = smf.negativebinomial(formula=formula, data=df_recent).fit(disp=0)
    coef = fit.params['VAR']
    pval = fit.pvalues['VAR']
    irr = np.exp(coef)
    glm_recent_results.append({
        'Variable': name,
        'VAR Coefficient': round(coef, 4),
        'Incidence Rate Ratio': round(irr, 3),
        'P-Value': round(pval, 4),
        'Significant': 'Yes' if pval < 0.05 else 'No'
    })
glm_recent_df = pd.DataFrame(glm_recent_results)
print(glm_recent_df.to_string(index=False))

# --- Multinomial logistic regression (recent window) ---
print("\nMultinomial Logistic Regression: Match Outcome ~ VAR (recent-seasons window)")
print("-" * 70)
df_recent['Result_Code'] = df_recent['FT Result'].map({'D': 0, 'H': 1, 'A': 2})
X_recent = sm.add_constant(df_recent['VAR'])
y_recent = df_recent['Result_Code']
mnlogit_recent_fit = sm.MNLogit(y_recent, X_recent).fit(disp=0)

odds_ratios_recent = np.exp(mnlogit_recent_fit.params)
odds_ratios_recent.columns = ['Home Win vs Draw', 'Away Win vs Draw']
pvals_recent = mnlogit_recent_fit.pvalues
pvals_recent.columns = ['Home Win vs Draw', 'Away Win vs Draw']

print("Odds Ratios (relative to Draw baseline):")
print(odds_ratios_recent)
print("\nP-values:")
print(pvals_recent)

# --- Comparison summary: full-era vs recent-window significance ---
print("\n" + "=" * 70)
print("SUMMARY: Full-era vs Recent-window significance comparison")
print("=" * 70)
comparison_summary = pd.DataFrame({
    'Variable': results_df['Variable'],
    'Significant (full era, 2000-2024)': results_df['Significant (both tests)'],
    'Significant (recent window, 2014-2024)': results_recent_df['Significant (both tests)']
})
print(comparison_summary.to_string(index=False))

In [ ]:
# ============================================================
# REPLACEMENT for Cell 13 — dual-test comparison chart
# Works with the new results_df structure from Cell A
# (columns: Welch t P-value, Mann-Whitney U P-value, Significant (both tests))
# ============================================================
fig, ax = plt.subplots(figsize=(12, 7))

variables_list = list(results_df['Variable'])
welch_p = list(results_df['Welch t P-value'])
mw_p = list(results_df['Mann-Whitney U P-value'])
sig = list(results_df['Significant (both tests)'])

y_pos = np.arange(len(variables_list))
bar_height = 0.35

# Use -log10(p-value) so small p-values show as LARGE bars
welch_log = [-np.log10(max(p, 0.0001)) for p in welch_p]
mw_log = [-np.log10(max(p, 0.0001)) for p in mw_p]

ax.barh(y_pos + bar_height/2, welch_log, height=bar_height,
        label="Welch's t-test", color='#4472C4', edgecolor='black')
ax.barh(y_pos - bar_height/2, mw_log, height=bar_height,
        label='Mann-Whitney U-test', color='#ED7D31', edgecolor='black')

# Threshold line at -log10(0.05) = 1.301
ax.axvline(x=1.301, color='black', linestyle='--', linewidth=1.5,
           label='Significance threshold (p=0.05)')

ax.set_yticks(y_pos)
ax.set_yticklabels(variables_list)
ax.set_xlabel('-log10(P-Value)  |  Larger bar = More significant')
ax.set_title("Statistical Testing Results: Pre-VAR vs Post-VAR\n(Welch's t-test vs Mann-Whitney U-test)")
ax.legend(loc='lower right')

# Mark significant (both tests) variables
for i, s in enumerate(sig):
    mark = '\u2713' if s == 'Yes' else '\u2717'
    ax.text(max(welch_log[i], mw_log[i]) + 0.15, i, mark, va='center',
            fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('statistical_testing_results.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Chart saved!")

## 6. Machine-learning evaluation

Train on pre-VAR matches and evaluate on post-VAR matches. Full-time goal variables are excluded from predictors to avoid target leakage.

In [ ]:
# Target variable encoding
target = 'FT Result'

# Encode target variable (H=0, D=1, A=2)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Result_Encoded'] = le.fit_transform(df[target])

print("Target encoding:")
print(dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# Remove goals - they cause data leakage
# Use only performance features that don't reveal the result
features = ['H Shots', 'A Shots',
            'H SOT', 'A SOT',
            'H Fouls', 'A Fouls',
            'H Corners', 'A Corners',
            'H Yellow', 'A Yellow',
            'H Red', 'A Red']

# Prepare features and target again
X_train = df[df['VAR']==0][features]
y_train = df[df['VAR']==0]['Result_Encoded']

X_test = df[df['VAR']==1][features]
y_test = df[df['VAR']==1]['Result_Encoded']

print("Updated features:", features)
print("Total features:", len(features))
print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

In [ ]:
# Retrain all 5 models with corrected features
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define all 5 models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss'),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(64, 32),
                                         max_iter=500, random_state=42)
}

# Train and evaluate each model
model_results = []

for name, model in models.items():
    print(f"Training {name}...")

    # Use scaled data for Logistic Regression and MLP
    if name in ['Logistic Regression', 'MLP Neural Network']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    model_results.append({
        'Model': name,
        'Test Accuracy': round(accuracy * 100, 2),
        'F1 Score': round(f1, 3),
        'CV Mean Accuracy': round(cv_mean * 100, 2),
        'CV Std': round(cv_std * 100, 2)
    })

    print(f"  Test Accuracy: {accuracy*100:.2f}%")
    print(f"  F1 Score: {f1:.3f}")
    print(f"  CV Accuracy: {cv_mean*100:.2f}% (+/- {cv_std*100:.2f}%)")
    print()

model_results_df = pd.DataFrame(model_results)
print("=" * 60)
print("FINAL MODEL COMPARISON")
print("=" * 60)
print(model_results_df.to_string(index=False))

In [ ]:
# Visualise model comparison
fig, ax = plt.subplots(figsize=(12, 6))

model_names = model_results_df['Model']
test_acc = model_results_df['Test Accuracy']
cv_acc = model_results_df['CV Mean Accuracy']

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, test_acc, width,
               label='Test Accuracy (Post-VAR)', color='steelblue')
bars2 = ax.bar(x + width/2, cv_acc, width,
               label='CV Accuracy (Pre-VAR)', color='coral')

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Performance Comparison\nTrained on Pre-VAR Data, Tested on Post-VAR Data')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 80)
ax.axhline(y=33.3, color='red', linestyle='--',
           linewidth=1, label='Random baseline (33.3%)')

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print("Chart saved!")

## 7. Model interpretation with SHAP

Estimate feature contributions for a Random Forest model using a sample of post-VAR matches.

In [ ]:
import shap

# Use a simpler faster approach - use only 200 sample rows for SHAP
X_test_sample = X_test.sample(200, random_state=42)

# Train a smaller Random Forest just for SHAP
from sklearn.ensemble import RandomForestClassifier
rf_shap = RandomForestClassifier(n_estimators=20, random_state=42)
rf_shap.fit(X_train, y_train)

# Create SHAP explainer on smaller model and sample
explainer = shap.TreeExplainer(rf_shap)
shap_values = explainer.shap_values(X_test_sample)

print("SHAP values computed successfully")
print("Type:", type(shap_values))
if isinstance(shap_values, list):
    print("Number of classes:", len(shap_values))
    print("Shape per class:", shap_values[0].shape)
else:
    print("Shape:", shap_values.shape)

In [ ]:
import matplotlib.pyplot as plt

# Class labels for reference
class_names = ['Away Win', 'Draw', 'Home Win']

# --- Option 1: Overall feature importance (mean |SHAP| across all classes) ---
# Average across the class dimension, then across samples
mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))  # shape: (12,)

importance_df = pd.DataFrame({
    'Feature': features,
    'Mean |SHAP value|': mean_abs_shap
}).sort_values('Mean |SHAP value|', ascending=False)

print(importance_df.to_string(index=False))

plt.figure(figsize=(8, 6))
plt.barh(importance_df['Feature'], importance_df['Mean |SHAP value|'])
plt.xlabel('Mean |SHAP value|')
plt.title('Overall Feature Importance (Random Forest, SHAP)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('shap_overall_importance.png', dpi=150)
plt.show()

# --- Option 2: Summary plot for one class at a time (e.g. Home Win) ---
shap.summary_plot(shap_values[:, :, 2], X_test_sample, feature_names=features,
                   show=False)
plt.title('SHAP Summary — Home Win')
plt.tight_layout()
plt.savefig('shap_home_win_summary.png', dpi=150)
plt.show()

## 8. Deep neural network

Scale the features and evaluate the DNN on the held-out post-VAR period. Compare accuracy alongside weighted F1 rather than relying on accuracy alone.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Scale features (already have scaler from earlier, but refit to be safe)
scaler_dl = StandardScaler()
X_train_dl = scaler_dl.fit_transform(X_train)
X_test_dl = scaler_dl.transform(X_test)

# Build a proper deep neural network
model_dl = keras.Sequential([
    layers.Input(shape=(X_train_dl.shape[1],)),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(3, activation='softmax')  # 3 classes: Away, Draw, Home
])

model_dl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_dl.summary()

# Early stopping to prevent overfitting
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

# Train with a validation split (from training data only - test set stays untouched)
history = model_dl.fit(
    X_train_dl, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate on the post-VAR test set
y_pred_probs = model_dl.predict(X_test_dl)
y_pred_dl = np.argmax(y_pred_probs, axis=1)

accuracy_dl = accuracy_score(y_test, y_pred_dl)
f1_dl = f1_score(y_test, y_pred_dl, average='weighted')

print(f"\nDeep Learning Model Results:")
print(f"Test Accuracy: {accuracy_dl*100:.2f}%")
print(f"F1 Score: {f1_dl:.3f}")

# Add to your results table (using model_results, the correct clean list)
model_results.append({
    'Model': 'Deep Neural Network',
    'Test Accuracy': round(accuracy_dl * 100, 2),
    'F1 Score': round(f1_dl, 3),
    'CV Mean Accuracy': None,  # k-fold CV not applied to DL training (see note)
    'CV Std': None
})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Deep Neural Network — Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Deep Neural Network — Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('dnn_training_curves.png', dpi=150)
plt.show()

In [ ]:
results_df_final = pd.DataFrame(model_results)

# Sort by Test Accuracy, descending
results_df_final = results_df_final.sort_values('Test Accuracy', ascending=False).reset_index(drop=True)

print("=" * 65)
print("FINAL MODEL COMPARISON — ALL 6 MODELS (leak-free features)")
print("=" * 65)
print(results_df_final.to_string(index=False))

# Save as CSV for easy inclusion in your dissertation
results_df_final.to_csv('final_model_comparison.csv', index=False)
print("\nSaved to final_model_comparison.csv")

## 9. Robustness and sensitivity checks

Check whether the main patterns persist in a balanced recent-seasons window and when the COVID-affected 2020/21 season is excluded.

In [ ]:
# Robustness Check 1: Recent-seasons-only split (balanced pre/post window)
recent_pre_var_seasons = ['2014/15', '2015/16', '2016/17', '2017/18', '2018/19']
recent_post_var_seasons = ['2019/20', '2020/21', '2021/22', '2022/23', '2023/24']

X_train_recent = df[df['Season'].isin(recent_pre_var_seasons)][features]
y_train_recent = df[df['Season'].isin(recent_pre_var_seasons)]['Result_Encoded']

X_test_recent = df[df['Season'].isin(recent_post_var_seasons)][features]
y_test_recent = df[df['Season'].isin(recent_post_var_seasons)]['Result_Encoded']

print("Recent-seasons training set:", X_train_recent.shape)
print("Recent-seasons testing set:", X_test_recent.shape)

# Scale
scaler_recent = StandardScaler()
X_train_recent_scaled = scaler_recent.fit_transform(X_train_recent)
X_test_recent_scaled = scaler_recent.transform(X_test_recent)

# Retrain all 5 classical models on this balanced window
recent_split_results = []

for name, model_class in [
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42)),
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('XGBoost', XGBClassifier(random_state=42, eval_metric='mlogloss')),
    ('MLP Neural Network', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
]:
    print(f"Training {name} (recent-seasons split)...")

    if name in ['Logistic Regression', 'MLP Neural Network']:
        model_class.fit(X_train_recent_scaled, y_train_recent)
        y_pred = model_class.predict(X_test_recent_scaled)
    else:
        model_class.fit(X_train_recent, y_train_recent)
        y_pred = model_class.predict(X_test_recent)

    accuracy = accuracy_score(y_test_recent, y_pred)
    f1 = f1_score(y_test_recent, y_pred, average='weighted')

    recent_split_results.append({
        'Model': name,
        'Test Accuracy': round(accuracy * 100, 2),
        'F1 Score': round(f1, 3)
    })
    print(f"  Test Accuracy: {accuracy*100:.2f}%  F1: {f1:.3f}")

recent_split_df = pd.DataFrame(recent_split_results)
print("\n" + "=" * 60)
print("ROBUSTNESS CHECK 1: RECENT-SEASONS-ONLY SPLIT")
print("=" * 60)
print(recent_split_df.to_string(index=False))

In [ ]:
# Robustness Check 2: COVID-adjusted sensitivity test (exclude 2020/21)
df_no_covid = df[df['COVID'] == 0]

X_train_covidadj = df_no_covid[df_no_covid['VAR']==0][features]
y_train_covidadj = df_no_covid[df_no_covid['VAR']==0]['Result_Encoded']

X_test_covidadj = df_no_covid[df_no_covid['VAR']==1][features]
y_test_covidadj = df_no_covid[df_no_covid['VAR']==1]['Result_Encoded']

print("COVID-excluded training set:", X_train_covidadj.shape)
print("COVID-excluded testing set:", X_test_covidadj.shape)

scaler_covidadj = StandardScaler()
X_train_covidadj_scaled = scaler_covidadj.fit_transform(X_train_covidadj)
X_test_covidadj_scaled = scaler_covidadj.transform(X_test_covidadj)

covid_adj_results = []

for name, model_class in [
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42)),
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('XGBoost', XGBClassifier(random_state=42, eval_metric='mlogloss')),
    ('MLP Neural Network', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
]:
    print(f"Training {name} (COVID-adjusted)...")

    if name in ['Logistic Regression', 'MLP Neural Network']:
        model_class.fit(X_train_covidadj_scaled, y_train_covidadj)
        y_pred = model_class.predict(X_test_covidadj_scaled)
    else:
        model_class.fit(X_train_covidadj, y_train_covidadj)
        y_pred = model_class.predict(X_test_covidadj)

    accuracy = accuracy_score(y_test_covidadj, y_pred)
    f1 = f1_score(y_test_covidadj, y_pred, average='weighted')

    covid_adj_results.append({
        'Model': name,
        'Test Accuracy': round(accuracy * 100, 2),
        'F1 Score': round(f1, 3)
    })
    print(f"  Test Accuracy: {accuracy*100:.2f}%  F1: {f1:.3f}")

covid_adj_df = pd.DataFrame(covid_adj_results)
print("\n" + "=" * 60)
print("ROBUSTNESS CHECK 2: COVID-ADJUSTED SENSITIVITY TEST")
print("=" * 60)
print(covid_adj_df.to_string(index=False))